## **📄 Clean & Jerk Dataset — Publication Figures & Tables**

### 🔧 Colab Setup & Library Init

In [2]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import (
    DEFAULT_CLEAN_DATASET_DIR,
    build_annotation_agreement_table,
    build_imu_quality_table,
    build_paper_tables,
    plot_fig4_annotation_scheme,
    plot_all_sets_annotation_sd_heatmap,
    plot_annotation_agreement_figure,
    plot_imu_validation_figure,
    plot_set_annotator_grid,
    build_static_vs_quasistatic_table,
    plot_quasistatic_imu_validation,
    plot_stationary_gyro_bias_profile,
)

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
EXPORT_CFG = {"toImageButtonOptions": {"format": "png", "scale": 4}}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ImportError: cannot import name 'build_static_vs_quasistatic_table' from 'src.explorer' (/content/Wearable-Biomechanics/src/explorer/__init__.py)

### 📊 Paper Tables (Tables 5, 7, and 8)

**Mathematical & Structural Definitions:**
- **Table 5 (Participant Demographics & Recording Volume):** Merges participant metadata with per-participant recording totals: $\text{Sets}_p = |\{s : \text{pid}(s)=p\}|$ and $\text{Lifts}_p = \sum_{s \in p} N_{\text{lifts}}(s)$.
- **Table 7 (Dataset Coverage & Fault Stratification):** Partitions all sets into **Complete (No Faults)** and **Fault-Tolerant** tiers using `has_faults`, reporting total recording duration $T_{\text{min}} = \frac{1}{60}\sum_s T_s$ and cumulative IMU sample count $\sum_s N_s$.
- **Table 8 (Phase Boundary Inter-Annotator Agreement):** For each of the 12 phase markers $m$, summarizes the sample standard deviation of frame indices across annotators ($s_f$ in frames, $s_{\text{ms}}$ in ms), reporting $\overline{s_f}$, $\overline{s_{\text{ms}}}$, $\max(s_f)$, and boundary-tolerance pass rates $\%(s_f \le 6)$ and $\%(s_f \le 8)$ frames.

> **Why this is done:** Separating fault-free from fault-tolerant sets demonstrates total usable data volume transparently, while frame-threshold pass rates contextualize annotation precision against the video sampling resolution.

In [ ]:
# @title 📊 Generate Paper Tables 5, 7, 8
paper_tables = build_paper_tables(dataset_dir)
for name, df in paper_tables.items():
    print(f"\n=== {name} ===")
    display_table(df, clean_columns=False)


=== table5_demographics ===


   participant_id                       name  age_years sex  height_m  \
0           P0001                      Enuka        NaN   M       NaN   
1           P0002      Thevindu Samarasekera       25.0   M      1.69   
2           P0003             Achalani Royal       23.0   F      1.60   
3           P0004                    Pandula        NaN   M       NaN   
4           P0005          Induwara Liyanage       21.0   M      1.70   
5           P0006                   Duleetha        NaN   M       NaN   
6           P0007         Hivindu Kirulapana       22.0   M      1.74   
7           P0008          Yewin Dissanayake       22.0   M      1.70   
8           P0009         Yasith Dissanayake       22.0   M      1.78   
9           P0010  J W K Mahasen Jayasundara       24.0   M      1.78   
10          P0011      Nirmal Samarawickrama       22.0   M      1.73   
11          P0012   J. M. Sudeshe Jayasinghe       23.0   M      1.56   
12          P0013          Dulan Jayasooriya       


=== table7_coverage ===


                   tier  sets  participants  lifts_total  lifts_usable  \
0  Complete (No Faults)    35            15          157           157   
1        Fault-Tolerant    19             9           89            80   
2                 Total    54            19          246           237   

   duration_min  total_imu_samples  
0         44.37             266229  
1         25.36             152162  
2         69.73             418391  


=== table8_agreement ===


          marker_name  n_boundaries  mean_sd_frames  mean_sd_ms  \
0               start           246            2.64        88.0   
1               setup           246            2.79        93.2   
2          first_pull           246            2.13        70.9   
3         second_pull           246            1.51        50.4   
4               clean           246            2.27        75.5   
5   intermediate_rest           246            3.11       103.5   
6          jerk_split           246            2.27        75.7   
7       jerk_recovery           246            2.85        95.1   
8       overhead_hold           246            3.37       112.3   
9             descend           246            2.51        83.8   
10          post_rest           246            3.38       112.6   
11                end           246            2.05        68.4   

    max_sd_frames  pct_within_6_frames  pct_within_8_frames  
0            7.51                 90.7                100.0  
1   

### 🖼️ 11-Phase Annotation Scheme

**Mathematical Formulation:**
Each repetition (lift) is partitioned by 12 ordered consensus event timestamps $t_1 < t_2 < \dots < t_{12}$ (`start` through `end`) into 11 contiguous biomechanical phases $P_k = [t_k, t_{k+1}]$ for $k = 1, \dots, 11$. For any tri-axial IMU signal $\mathbf{x}(t) = [x_X(t), x_Y(t), x_Z(t)]^\top$, the orientation-invariant Euclidean norm is computed as:

$$\|\mathbf{a}(t)\|_2 = \sqrt{a_x(t)^2 + a_y(t)^2 + a_z(t)^2} \quad (\text{m/s}^2), \qquad \|\boldsymbol{\omega}(t)\|_2 = \sqrt{\omega_x(t)^2 + \omega_y(t)^2 + \omega_z(t)^2} \quad (\text{deg/s})$$

> **Why this is done:** Plotting consensus phase boundaries directly over tri-axial waveforms (and/or their Euclidean norm) visually verifies that kinematic inflection points (e.g., `first_pull`, `second_pull`, `jerk_split`, `overhead_hold`) align with the labeled biomechanical phases.

In [73]:
# @title 🖼️ Figure 4: Annotation Scheme { display-mode: "form" }
sample_set_id = "set_0052" # @param {type:"string"}
sample_lifts = "2-5" # @param {type:"string"}  # e.g. "1", "1,2", "1-5", or "all"
sample_sensors = "RHP" # @param {type:"string"}  # e.g. "RHD" or "RKP,RHD" or "US,LS"
modality = "both" # @param ["gyro", "accel", "both"]
signal_mode = "axes" # @param ["axes", "magnitude", "both"]
axes_str = "X,Y,Z" # @param {type:"string"}  # e.g. "X,Y,Z" or "Y" or "X,Z"

fig4 = plot_fig4_annotation_scheme(
    dataset_dir,
    set_id=sample_set_id,
    lift_ids=sample_lifts,
    sensor_codes=sample_sensors,
    modality=modality,
    signal_mode=signal_mode,
    axes=axes_str,
)
fig4.show(config=EXPORT_CFG)

### 📊 IMU Technical Validation Summary Table

**Mathematical Definitions (computed across all $54 \times 26 = 1{,}404$ set–sensor recordings):**
Let $\mathcal{S}$ denote the **static rest window** of a set, formed by the union of all `start` $\to$ `setup` and `post_rest` $\to$ `end` intervals (requiring $> 10$ samples; fallback to the first $\min(100, N)$ samples if absent):

1. **Data Completeness ($\%$):** Percentage of samples where all 6 IMU channels $(a_x, a_y, a_z, \omega_x, \omega_y, \omega_z)$ are finite (non-NaN). Any recording with zero variance ($\sigma = 0$) on any axis is flagged as flatlined ($0\%$ completeness). Missing/flatlined $\% = 100\% - \text{Completeness}$.
2. **Static Accelerometer Norm ($\text{m/s}^2$):** Mean Euclidean acceleration magnitude during stationary rest $\mathcal{S}$ (for recordings with completeness $> 50\%$):
   $$\bar{a}_{\text{static}} = \frac{1}{|\mathcal{S}|} \sum_{t \in \mathcal{S}} \sqrt{a_x(t)^2 + a_y(t)^2 + a_z(t)^2} \approx g = 9.81\ \text{m/s}^2$$
   *(Pass criterion: $|\bar{a}_{\text{static}} - 9.81| \le 0.50\ \text{m/s}^2$)*
3. **Static Gyroscope Bias Magnitude ($\text{deg/s}$):** Mean Euclidean angular-velocity magnitude during stationary rest $\mathcal{S}$:
   $$\bar{\omega}_{\text{static}} = \frac{1}{|\mathcal{S}|} \sum_{t \in \mathcal{S}} \sqrt{\omega_x(t)^2 + \omega_y(t)^2 + \omega_z(t)^2} \approx 0\ \text{deg/s}$$
   *(Pass criterion: $\bar{\omega}_{\text{static}} < 5.0\ \text{deg/s}$)*
4. **Sensor Saturation ($\%$):** Fraction of samples exceeding full-scale bounds ($\max_j |a_j(t)| > 80\ \text{m/s}^2$ or $\max_j |\omega_j(t)| > 2000\ \text{deg/s}$).
5. **Inter-IMU Synchronization Lag (ms):** Normalized dynamic acceleration magnitude $\tilde{a}(t) = \|\mathbf{a}(t)\|_2 - \text{median}(\|\mathbf{a}\|_2)$ (standardized to zero mean, unit variance) cross-correlated against the Upper Spine (`US`) reference within $\pm 200\ \text{ms}$:
   $$\Delta t = \frac{1000}{f_s} \cdot \arg\max_{\tau \in [-L, L]} \sum_t \tilde{a}_{\text{US}}(t)\,\tilde{a}_s(t + \tau)$$

> **Why this is done:** Using Euclidean norms during annotated static rest windows evaluates factory calibration, zero-rate bias, and sensor integrity independently of sensor-to-segment orientation. Reporting both Mean $\pm$ SD and Median $[\text{IQR}]$ alongside pass rates captures both central tendency and sensitivity to rare outliers.

In [ ]:
# @title 📊 IMU Quality Summary Table (Option 1)
df_imu_quality = build_imu_quality_table(dataset_dir)
display_table(df_imu_quality, clean_columns=False)

📝 Draft text: "Across all 1,404 sensor recordings (54 sets × 26 IMUs), data completeness was 97.4 ± 15.8% (min 0.0%). The accelerometer norm during static periods was 9.67 ± 1.27 m/s², and static gyroscope bias was 10.29 ± 8.87 deg/s."


                                    Metric Expected / Threshold  \
0      Data completeness (% valid samples)                > 99%   
1          Missing / flatlined samples (%)                 < 1%   
2         Static accelerometer norm (m/s²)     9.81 ± 0.50 m/s²   
3  Static gyroscope bias magnitude (deg/s)          < 5.0 deg/s   
4   Accelerometer saturation (% > 80 m/s²)        ≈ 0% (< 0.1%)   
5    Gyroscope saturation (% > 2000 deg/s)        ≈ 0% (< 0.1%)   
6        Inter-IMU dynamic lag vs. US (ms)   < 10 ms (1 sample)   

        Mean ± SD             Median [IQR]      Min – Max  \
0   97.43 ± 15.81  100.00 [100.00, 100.00]  0.00 – 100.00   
1    2.57 ± 15.81        0.00 [0.00, 0.00]  0.00 – 100.00   
2     9.67 ± 1.27       9.81 [9.62, 10.06]   0.00 – 12.39   
3    10.29 ± 8.87       7.90 [5.35, 12.13]  0.00 – 125.50   
4     0.00 ± 0.00        0.00 [0.00, 0.00]    0.00 – 0.00   
5     0.00 ± 0.00        0.00 [0.00, 0.00]    0.00 – 0.00   
6  104.64 ± 73.98   100.00 [30.00, 1

### 🖼️ IMU Technical Validation Figure (Static Accel Norm & Gyro Bias)

**Mathematical Basis & Outlier-Zoom Strategy:**
- **Panel (a) — Static Accelerometer Norm per IMU:** Plots the distribution of $\bar{a}_{\text{static}}$ across all sets for each of the 26 sensors, with a dashed reference at standard gravity ($g = 9.81\ \text{m/s}^2$).
- **Panel (b) — Static Gyroscope Bias per IMU:** Plots the distribution of $\bar{\omega}_{\text{static}}$ across all sets per sensor, with a dashed reference at $0\ \text{deg/s}$.
- **Outlier Zoom Control (`show_outliers`):** Because occasional flatlined recordings ($0\ \text{m/s}^2$) or brief pre-lift movement in a static window ($> 50\ \text{deg/s}$) compress the y-axis, `show_outliers=False` hides extreme dots and sets IQR-derived bounds:
  $$y_{\min} = Q_1 - 2.0 \cdot \text{IQR}, \qquad y_{\max} = Q_3 + 2.5 \cdot \text{IQR}$$
  (floored at $0$ for gyroscope bias and padded slightly for readability). Toggle `show_outliers=True` to inspect all raw outliers.

> **Why this is done:** Per-sensor boxplots verify that calibration accuracy and gyroscope bias are consistent across all 26 anatomical placements rather than masking a faulty sensor in a pooled average.

In [ ]:
# @title 🖼️ IMU Technical Validation Figure (Static Accel Norm & Gyro Bias)
show_outliers = False # @param {type:"boolean"}
fig_val = plot_imu_validation_figure(dataset_dir, show_outliers=show_outliers)
fig_val.show(config=EXPORT_CFG)

### 🔥 Per-Set Annotator Grid & SD (ms) Heatmap

**Mathematical Formulation:**
For each lift $i$ and phase marker $m$ within the selected set, let $f_{i,m}^{(a)}$ and $t_{i,m}^{(a)}$ denote the frame index and timestamp (in seconds) marked by annotator $a \in \{1, \dots, K\}$ (typically $K = 3$):
- **Consensus Mean Frame:** $\bar{f}_{i,m} = \frac{1}{K} \sum_{a=1}^{K} f_{i,m}^{(a)}$
- **Inter-Annotator Sample SD (ms):**
  $$s_{i,m} = \sqrt{\frac{1}{K - 1} \sum_{a=1}^{K} \left(t_{i,m}^{(a)} - \bar{t}_{i,m}\right)^2} \times 1000\ \text{ms}$$

> **Why this is done:** Displaying raw annotator frames and their mean in uncolored rows while applying a blue-to-red color scale ($0\text{–}300\ \text{ms}$) exclusively to the SD rows lets reviewers inspect exact frame picks and immediately spot which specific repetition and phase boundary caused disagreement.

In [76]:
# @title 🔥 Per-Set Annotator Grid { display-mode: "form" }
ann_set_id = "set_0021" # @param {type:"string"}
fig_ann_set = plot_set_annotator_grid(dataset_dir, set_id=ann_set_id)
fig_ann_set.show(config=EXPORT_CFG)

### 🔥 All-Sets Inter-Annotator SD (ms) Heatmap

**Mathematical Formulation:**
Extends the per-set sample SD $s_{i,m}$ (in ms, using $n - 1$ Bessel's correction across the $K=3$ annotators) into a global matrix of all $(\text{set\_id}, \text{lift\_id})$ repetitions (rows) $\times$ 12 phase markers (columns), mapped to the unified $0\text{–}300\ \text{ms}$ blue-to-red color scale.

> **Why this is done:** Visualizes the entire dataset's annotation consistency in one view, distinguishing isolated single-repetition anomalies from systematic marker-level ambiguity across participants and sets.

In [77]:
# @title 🔥 All-Sets Inter-Annotator SD (ms) Heatmap
fig_ann_all = plot_all_sets_annotation_sd_heatmap(dataset_dir)
fig_ann_all.show(config=EXPORT_CFG)

### 📊 Annotation Agreement Summary Table (Right-Skewed)

**Mathematical & Statistical Rationale:**
1. **Presence Agreement:** Reports the percentage of marker instances identified by all $3/3$ annotators prior to timing analysis, because sample SD $s_{i,m}$ is only meaningful when all 3 annotators marked the event.
2. **Sample Standard Deviation (ms):** Computed with $n - 1$ denominator ($K = 3$ annotators) for every $3/3$-marked instance, and contextualized against the IMU sampling interval ($10\ \text{ms} = 1\ \text{sample at } 100\ \text{Hz}$).
3. **Non-Parametric Summary for Right-Skewed Data:** Because $s_{i,m} \ge 0$, the distribution is **right-skewed** (most instances cluster near $0\ \text{ms}$ with a long positive tail). Consequently, alongside Mean $\pm$ SD, the table reports **Median $[Q_1, Q_3]$**, **90th and 95th percentiles**, **Max (ms)**, and **$\% < \text{threshold}$** (default $200\ \text{ms}$).

> **Why this is done:** Reporting mean $\pm$ SD alone on right-skewed timing data overstates typical disagreement due to a few tail outliers; median, IQR, and upper percentiles accurately summarize both typical precision and worst-case spread for the manuscript.

In [78]:
# @title 📊 Annotation Agreement Table { display-mode: "form" }
threshold_ms = 200.0 # @param {type:"number"}  # 200 ms = 6 frames @ 30 fps (20 samples @ 100 Hz)
df_ann_table = build_annotation_agreement_table(dataset_dir, threshold_ms=threshold_ms)
display_table(df_ann_table, clean_columns=False)

📝 Draft text: "Of 2,952 marker instances, 100.0% were identified by all three annotators. For these, the inter-annotator SD of marker timestamps was right-skewed, with a median of 69.4 ms [IQR 33.3–126.2 ms]; 90% of instances showed an SD below 176.4 ms (≈ 17.6 samples at 100 Hz). Agreement was highest for first_pull (median 50.9 ms) and lowest for post_rest (median 117.1 ms)."


          Marker type  n instances Marked by all 3 (%) Median SD (ms) [IQR]  \
0               start          246              100.0%   69.4 [19.2, 153.7]   
1               setup          246              100.0%   83.9 [50.9, 134.7]   
2          first_pull          246              100.0%   50.9 [33.3, 100.0]   
3         second_pull          246              100.0%    50.9 [33.3, 66.7]   
4               clean          246              100.0%   69.4 [38.5, 101.8]   
5   intermediate_rest          246              100.0%   98.1 [50.9, 149.1]   
6          jerk_split          246              100.0%   66.7 [38.5, 100.0]   
7       jerk_recovery          246              100.0%   83.9 [50.9, 145.3]   
8       overhead_hold          246              100.0%  107.2 [52.6, 164.4]   
9             descend          246              100.0%   69.4 [38.5, 117.1]   
10          post_rest          246              100.0%  117.1 [50.9, 167.8]   
11                end          246              100.

### 🖼️ Pooled Inter-Annotator SD Distribution (Normalized)

**Mathematical Formulation:**
Pools $s_{i,m}$ (in ms) across all $3/3$-annotated marker instances and constructs a probability-normalized histogram (`histnorm="probability"`) with configurable bin width $\Delta w$ (`bin_width_ms`), where each bin $b$ has height:

$$p_b = \frac{n_b}{N_{\text{all3}}}, \qquad \sum_b p_b = 1$$

Vertical reference lines mark the empirical **Median ($Q_2$, 50th pct)**, **75th percentile ($Q_3$)**, **95th percentile**, and **99th percentile**.

> **Why this is done:** Normalizing bin heights to sum to 1 makes the distribution invariant to total instance count, while overlaying $Q_2$, $Q_3$, 95th, and 99th percentiles directly illustrates the right-skewed concentration near $0\ \text{ms}$ and quantifies the tail thresholds.

In [80]:
# @title 🖼️ Pooled Inter-Annotator SD Distribution { display-mode: "form" }
bin_width_ms = 15 # @param [10, 20, 50, 100] {type:"raw"}
fig_ann_agree = plot_annotation_agreement_figure(dataset_dir, bin_width_ms=bin_width_ms)
fig_ann_agree.show(config=EXPORT_CFG)